In [70]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [71]:
# Relative imports
d = os.path.abspath(os.getcwd())
os.chdir("../..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir(d)

### Read (and compact) dataframes

In [72]:
compact = True

In [73]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
data_dir = os.path.join(d, "..", "data")
for file in os.listdir(data_dir):
    fname = os.path.join(data_dir, file)
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(fname)
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(fname, index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(os.path.join(data_dir, f"combined_{timestamp}.parquet"))
    
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = os.path.join(data_dir, "trash")
    for f in read:
        os.rename(os.path.join(data_dir, f), os.path.join(trash, f))

dfs = []  # Clear memory
raw_df

Reading combined.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,...,player_type,opponent_names,action,amount,p,relative_ev,rank,tiebreakers,hand_index,state_id
state_id,,,,,,,,,,,,,,,,,,,,,
d2bfcb75-5c63-4875-8244-e93f3e968d9a,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],call,2,0.3629,0.010887,0,"[1, 0, 0, 0, 0]",75.0,NaN
479e187d-f0d1-495c-b3bb-796e28cccd45,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],raise,6,0.6223,0.018669,0,"[12, 8, 0, 0, 0]",554.0,NaN
13b12156-078e-4487-bcc6-8755bca2bb35,479e187d-f0d1-495c-b3bb-796e28cccd45,[],"[92, 87]",0,"[8, 13]","[8, 13]","[True, True]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],call,5,0.6223,0.065342,0,"[12, 8, 0, 0, 0]",554.0,NaN
c1ac65df-ef84-49ad-adc5-4042f7c61cff,13b12156-078e-4487-bcc6-8755bca2bb35,"[40, 43, 26]","[87, 87]",0,"[0, 0]","[13, 13]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],check,0,0.4384,0.056992,0,"[12, 8, 4, 1, 0]",554.0,NaN
77be572c-57b6-4b30-95ab-42f176f49a04,c1ac65df-ef84-49ad-adc5-4042f7c61cff,"[40, 43, 26, 23]","[87, 87]",0,"[0, 0]","[13, 13]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],check,0,0.3978,0.051714,0,"[12, 10, 8, 4, 1]",554.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2c1b0b52-cac7-4005-9688-f76deb94ca3f,095a2e29-170e-47d7-af66-bd402a304229,"[1, 45, 22]","[88, 91]",0,"[0, 5]","[8, 13]","[False, True]","[False, False]",1,4,...,HumanPlayer,[Max Mekker],fold,0,0.3199,0.033590,0,"[9, 8, 6, 1, 0]",605.0,NaN
4a8ec0af-1ded-4a12-a0ba-2b6a9b6b013e,None,[],"[86, 108]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],call,2,0.5631,0.016893,0,"[10, 5, 0, 0, 0]",262.0,NaN
0b6fde39-0191-40c1-aa44-fbd32b7c7309,4a8ec0af-1ded-4a12-a0ba-2b6a9b6b013e,[],"[84, 103]",0,"[4, 9]","[4, 9]","[True, True]","[False, False]",0,4,...,HumanPlayer,[Max Mekker],call,5,0.5631,0.036601,0,"[10, 5, 0, 0, 0]",262.0,NaN


In [74]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
opponent_names        object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
hand_index           float64
state_id             float64
dtype: object

In [75]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
dupe_df

,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,...,player_type,opponent_names,action,amount,p,relative_ev,rank,tiebreakers,hand_index,state_id
state_id,,,,,,,,,,,,,,,,,,,,,


## Process data

In [76]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,game_id,action,amount,excess_rank,p,relative_ev,stage,player_name,opponent_name,n_players
d2bfcb75-5c63-4875-8244-e93f3e968d9a,0,0,0,0,0,0,0,0,0,0,...,49c246a2-cfc0-4aae-854a-37b56047526b,call,2,0,0.3629,0.010887,preflop,Tord,Max Mekker,2
479e187d-f0d1-495c-b3bb-796e28cccd45,0,0,0,0,0,0,0,0,0,0,...,5076c842-768a-41ff-a51a-6b0eb7dc83b8,raise,6,0,0.6223,0.018669,preflop,Tord,Max Mekker,2
13b12156-078e-4487-bcc6-8755bca2bb35,6,0,0,0,0,0,0,0,0,0,...,5076c842-768a-41ff-a51a-6b0eb7dc83b8,call,5,0,0.6223,0.065342,preflop,Tord,Max Mekker,2
c1ac65df-ef84-49ad-adc5-4042f7c61cff,6,0,0,0,0,5,0,0,0,0,...,5076c842-768a-41ff-a51a-6b0eb7dc83b8,check,0,0,0.4384,0.056992,flop,Tord,Max Mekker,2
77be572c-57b6-4b30-95ab-42f176f49a04,6,0,0,0,0,5,0,0,0,0,...,5076c842-768a-41ff-a51a-6b0eb7dc83b8,check,0,0,0.3978,0.051714,turn,Tord,Max Mekker,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2c1b0b52-cac7-4005-9688-f76deb94ca3f,0,0,0,0,0,4,0,0,0,0,...,4b45277c-eb12-4632-810c-8472b1a0bd3f,fold,0,0,0.3199,0.033590,flop,Tord,Max Mekker,2
4a8ec0af-1ded-4a12-a0ba-2b6a9b6b013e,0,0,0,0,0,0,0,0,0,0,...,67504bf1-2ba5-4775-8ca8-5d7a65001abc,call,2,0,0.5631,0.016893,preflop,Tord,Max Mekker,2
0b6fde39-0191-40c1-aa44-fbd32b7c7309,0,0,0,0,0,2,0,0,0,0,...,67504bf1-2ba5-4775-8ca8-5d7a65001abc,call,5,0,0.5631,0.036601,preflop,Tord,Max Mekker,2
9fd3794c-8f2d-42ec-9d6a-eaea7f103c96,0,0,0,0,0,7,0,0,0,0,...,67504bf1-2ba5-4775-8ca8-5d7a65001abc,check,0,0,0.4420,0.039780,flop,Tord,Max Mekker,2


In [77]:
df.dtypes

raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn          int64
opponent_check_river

## Training

In [78]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [79]:
X = df.drop(["game_id", "action", "amount"], axis=1)
y = df["action"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [80]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_check_turn,opponent_check_river,opponent_check_showdown,excess_rank,p,relative_ev,stage,player_name,opponent_name,n_players
d2bfcb75-5c63-4875-8244-e93f3e968d9a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.3629,0.010887,preflop,Tord,Max Mekker,2
479e187d-f0d1-495c-b3bb-796e28cccd45,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.6223,0.018669,preflop,Tord,Max Mekker,2
13b12156-078e-4487-bcc6-8755bca2bb35,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.6223,0.065342,preflop,Tord,Max Mekker,2
c1ac65df-ef84-49ad-adc5-4042f7c61cff,6,0,0,0,0,5,0,0,0,0,...,0,0,0,0,0.4384,0.056992,flop,Tord,Max Mekker,2
77be572c-57b6-4b30-95ab-42f176f49a04,6,0,0,0,0,5,0,0,0,0,...,0,0,0,0,0.3978,0.051714,turn,Tord,Max Mekker,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2c1b0b52-cac7-4005-9688-f76deb94ca3f,0,0,0,0,0,4,0,0,0,0,...,0,0,0,0,0.3199,0.033590,flop,Tord,Max Mekker,2
4a8ec0af-1ded-4a12-a0ba-2b6a9b6b013e,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.5631,0.016893,preflop,Tord,Max Mekker,2
0b6fde39-0191-40c1-aa44-fbd32b7c7309,0,0,0,0,0,2,0,0,0,0,...,0,0,0,0,0.5631,0.036601,preflop,Tord,Max Mekker,2
9fd3794c-8f2d-42ec-9d6a-eaea7f103c96,0,0,0,0,0,7,0,0,0,0,...,0,0,0,0,0.4420,0.039780,flop,Tord,Max Mekker,2


In [81]:
y

d2bfcb75-5c63-4875-8244-e93f3e968d9a     call
479e187d-f0d1-495c-b3bb-796e28cccd45    raise
13b12156-078e-4487-bcc6-8755bca2bb35     call
c1ac65df-ef84-49ad-adc5-4042f7c61cff    check
77be572c-57b6-4b30-95ab-42f176f49a04    check
                                        ...  
2c1b0b52-cac7-4005-9688-f76deb94ca3f     fold
4a8ec0af-1ded-4a12-a0ba-2b6a9b6b013e     call
0b6fde39-0191-40c1-aa44-fbd32b7c7309     call
9fd3794c-8f2d-42ec-9d6a-eaea7f103c96    check
21f0daec-c336-4b6a-8cbd-cc6216341d1e     fold
Name: action, Length: 4707, dtype: object

In [82]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["excess_rank", "stage", "player_name", "opponent_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [83]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (3778, 37)
Test shape: (929, 37)


In [84]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.6383207750269106


In [85]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.6383207750269106
Mean goodness:  0.5185395549684599


,call,check,fold,raise,true,pred,correct,goodness
0ba055c3-da1d-4eba-a2b8-e4d7cb262e97,0.654422,0.099239,9.046720e-02,0.155872,call,call,True,0.654422
68581669-c9e9-48ce-9c1c-84c6cfd8e4ef,0.769989,0.011950,1.794376e-01,0.038623,call,call,True,0.769989
7b7f46ed-c202-489b-99ec-931c313955ac,0.073450,0.183577,6.361233e-04,0.742337,raise,raise,True,0.742337
e8e25699-3170-459b-8015-a58517cd739e,0.020373,0.303914,5.957500e-07,0.675712,raise,raise,True,0.675712
1a07e5ea-94e5-4428-81a2-7c26f579b181,0.002819,0.327654,6.395430e-14,0.669527,call,raise,False,0.002819
...,...,...,...,...,...,...,...,...
865f17b9-417b-4852-bb56-4a743315700d,0.588058,0.128910,2.066287e-01,0.076404,call,call,True,0.588058
6a8097e9-2b7f-4ca0-a227-87dcccfe4c05,0.449902,0.289240,2.167131e-01,0.044145,check,call,False,0.289240
9584540f-ba48-4de2-9c80-53264252d81b,0.106087,0.198686,6.878219e-01,0.007405,check,fold,False,0.198686
7b35ddb5-2832-4f01-ab3c-53f8de4edb82,0.103022,0.370632,5.133471e-01,0.012999,check,fold,False,0.370632


In [86]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

336 incorrect predictions:


,call,check,fold,raise,true,pred,correct,goodness
1a07e5ea-94e5-4428-81a2-7c26f579b181,0.002819,0.327654,6.395430e-14,0.669527,call,raise,False,0.002819
ddb872bb-e2cd-4373-bb87-6e56626c4caf,0.507268,0.085531,3.913854e-01,0.015816,fold,call,False,0.391385
12d7a3ff-6d27-4d9d-89b6-ed8b32fd8a81,0.536088,0.286046,1.077448e-01,0.070122,check,call,False,0.286046
73bc5e57-f4fd-43a6-985c-10cfc1d89c81,0.228534,0.327153,4.201666e-01,0.024146,check,fold,False,0.327153
27ed3958-14f1-4c7c-8cb6-73ba922d85b4,0.456289,0.292232,2.057235e-01,0.045756,fold,call,False,0.205724
...,...,...,...,...,...,...,...,...
3a1f1543-f2b4-43f0-9ec7-8704896b93b7,0.016283,0.243870,7.382318e-14,0.739847,call,raise,False,0.016283
c5225e85-b7d8-4b96-893d-7071f9ddbc6c,0.031804,0.931529,4.394202e-06,0.036663,raise,check,False,0.036663
6a8097e9-2b7f-4ca0-a227-87dcccfe4c05,0.449902,0.289240,2.167131e-01,0.044145,check,call,False,0.289240
9584540f-ba48-4de2-9c80-53264252d81b,0.106087,0.198686,6.878219e-01,0.007405,check,fold,False,0.198686
